# Initial Pipeline Smoke Test

Deliberately tiny end-to-end pass through the **base** (`initial_pipeline`) forecasting and RL pipeline — the four base CL methods naive / ewc / replay / sdft (and recall for RL). Use it to catch integration errors and to exercise the SDFT convex-blend path before the expensive full run.

In [1]:
%load_ext autoreload
%autoreload 2

## Imports And Paths

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "initial_pipeline").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

from initial_pipeline.core_pipeline import CONFIG, configure_vast_ai, prepare_data, print_task_summary
from initial_pipeline.experiment_runner import (
    run_experiment,
    build_cl_summary,
    print_and_save_comparison_tables,
    generate_all_plots,
)
from initial_pipeline.trainers import LOGGER, compute_mase

Project root: C:\Users\Syakir\Downloads\Projects\fyp


C:\Users\Syakir\Downloads\Projects\fyp\.venv\Lib\site-packages\pytorch_forecasting\models\base\_base_model.py:30: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


## Runtime Setup

In [3]:
DATA_DIR = str(PROJECT_ROOT / "data" / "processed")
OUTPUT_DIR = str(PROJECT_ROOT / "outputs" / "initial_smoke_test")

configure_vast_ai(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    require_gpu=False,  # set True on Vast.ai if you want to require CUDA
)

Runtime diagnostics

Python         : 3.11.9

PyTorch        : 2.3.1+cu121

CUDA available : True

GPU            : NVIDIA GeForce RTX 3050 Laptop GPU

...

CPU cores      : 12

Vast.ai        : NO

Active device  : CUDA

Precision      : bf16-mixed

{'paths': {'demand_csv': 'C:\\Users\\Syakir\\Downloads\\Projects\\fyp\\data\\processed\\demand_forecasting.csv',
  'rl_csv': 'C:\\Users\\Syakir\\Downloads\\Projects\\fyp\\data\\processed\\rl_environment.csv',
  'checkpoints': 'C:\\Users\\Syakir\\Downloads\\Projects\\fyp\\outputs\\initial_smoke_test\\checkpoints',
  'results': 'C:\\Users\\Syakir\\Downloads\\Projects\\fyp\\outputs\\initial_smoke_test\\results',
  'logs': 'C:\\Users\\Syakir\\Downloads\\Projects\\fyp\\outputs\\initial_smoke_test\\logs',
  'plots': 'C:\\Users\\Syakir\\Downloads\\Projects\\fyp\\outputs\\initial_smoke_test\\plots'},
 'tasks': [{'task_id': 1,
   'name': 'Baseline_2023_H1',
   'start': '2023-01-01',
   'end': '2023-05-31',
   'regime': 'baseline'},
  {'task_id': 2,
   'name': 'MegaSale_2023',
   'start': '2023-06-01',
   'end': '2023-12-31',
   'regime': 'mega_sale'},
  {'task_id': 3,
   'name': 'Baseline_2024_H1',
   'start': '2024-01-01',
   'end': '2024-05-31',
   'regime': 'baseline'},
  {'task_id': 4,
   '

## Tiny Smoke-Test Configuration

In [4]:
# Keep this tiny. The goal is correctness, not final metrics.
CONFIG["tasks"] = CONFIG["tasks"][:2]
CONFIG["model_types"] = ["forecasting", "rl"]
CONFIG["cl_methods"] = {
    "forecasting": ["naive", "ewc", "replay", "sdft"],
    "rl": ["naive", "ewc", "recall", "sdft"],
}

CONFIG["forecasting"].update({
    "encoder_length": 28,
    "prediction_length": 7,
    "hidden_size": 16,
    "attention_head_size": 1,
    "hidden_continuous_size": 8,
    "batch_size": 64,
    "max_epochs": 1,
    "early_stop_patience": 1,
})

CONFIG["rl"].update({
    "total_timesteps_per_task": 256,
    "eval_episodes": 1,
    "n_steps": 128,
    "batch_size": 64,
    "n_epochs": 1,
    "net_arch": [32, 32],
})

CONFIG["cl"].update({
    "ewc_fisher_samples": 2,
    "replay_buffer_size": 128,
    "recall_buffer_capacity": 256,
    "recall_mix_n_steps": 32,
})

CONFIG["hardware"].update({
    "compile": False,
    "num_workers": 0,
    "persistent_workers": False,
})

print("Smoke-test config ready")
print("Tasks:", [t["name"] for t in CONFIG["tasks"]])
print("Forecast methods:", CONFIG["cl_methods"]["forecasting"])
print("RL methods:", CONFIG["cl_methods"]["rl"])

Smoke-test config ready
Tasks: ['Baseline_2023_H1', 'MegaSale_2023']
Forecast methods: ['naive', 'ewc', 'replay', 'sdft']
RL methods: ['naive', 'ewc', 'recall', 'sdft']


## Metric Sanity Check

In [5]:
mase_value = compute_mase([2, 3, 4], [2, 2, 5], list(range(20)), seasonality=7)
assert mase_value == mase_value and mase_value > 0, mase_value
print("MASE sanity check:", mase_value)

MASE sanity check: 0.09523809523809523


## Load Data

In [6]:
tft_tasks, rl_tasks, tft_df, rl_df = prepare_data()
print_task_summary(tft_tasks, rl_tasks)

assert len(tft_tasks) == len(CONFIG["tasks"])
assert len(rl_tasks) == len(CONFIG["tasks"])
assert all(len(df) > 0 for df in tft_tasks), "At least one TFT task is empty"
assert all(len(df) > 0 for df in rl_tasks), "At least one RL task is empty"
print("Data checks passed")

Loading datasets...

Demand CSV  : 9,864 rows × 24 cols

RL CSV      : 9,864 rows × 19 cols

✓ Data loaded and cleaned

┏━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┓
┃ Task ┃ Name             ┃ Period                   ┃ TFT rows ┃ RL rows ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━┩
│ 1    │ Baseline_2023_H1 │ 2023-01-01 -> 2023-05-31 │ 1,359    │ 1,359   │
│ 2    │ MegaSale_2023    │ 2023-06-01 -> 2023-12-31 │ 1,926    │ 1,926   │
└──────┴──────────────────┴──────────────────────────┴──────────┴─────────┘

Data checks passed


## Run Smoke Test

In [7]:
run_experiment(tft_tasks, rl_tasks)
print("Smoke-test training loop completed")

==============================================================

  CONTINUAL LEARNING EXPERIMENT START

==============================================================

═══ MODEL TYPE: FORECASTING ═══

  ── CL Method: naive ──

...

Eval task 1: avg_episode_reward=5961.8460  cumulative_profit=42518064.7445  pricing_regret=31.8379

Eval task 2: avg_episode_reward=7670.8176  cumulative_profit=43982609.9147  pricing_regret=21.8576

Results saved → C:\Users\Syakir\Downloads\Projects\fyp\outputs\initial_smoke_test\results\all_metrics.csv

═══ EXPERIMENT COMPLETE (2.8 min) ═══

Smoke-test training loop completed


## Validate Results

In [8]:
results_df = LOGGER.to_dataframe()
display(results_df.tail(20))

assert not results_df.empty, "No metrics were logged"
expected_model_types = set(CONFIG["model_types"])
assert expected_model_types.issubset(set(results_df["model_type"])), results_df["model_type"].unique()

forecast_df = results_df[results_df["model_type"] == "forecasting"]
rl_df_results = results_df[results_df["model_type"] == "rl"]
assert not forecast_df.empty, "No forecasting metrics logged"
assert not rl_df_results.empty, "No RL metrics logged"

# Confirm the SDFT (proposed) method actually ran on the forecasting side.
assert "sdft" in set(forecast_df["cl_method"]), forecast_df["cl_method"].unique()

# MASE can be NaN on a deliberately tiny split, but sMAPE/RMSE and RL metrics should exist.
assert {"smape", "rmse"}.issubset(set(forecast_df["metric_name"])), forecast_df["metric_name"].unique()
assert "cumulative_profit" in set(rl_df_results["metric_name"]), rl_df_results["metric_name"].unique()

print("Logged metrics:")
print(results_df.groupby(["model_type", "cl_method", "metric_name"]).size())
print("Smoke test passed")

,model_type,cl_method,train_task_id,eval_task_id,eval_phase,metric_name,metric_value,timestamp
92,rl,sdft,2,1,seen,pricing_regret,3.183790e+01,2026-06-03T23:03:59
93,rl,sdft,2,2,seen,avg_episode_reward,7.670818e+03,2026-06-03T23:04:01
94,rl,sdft,2,2,seen,cumulative_profit,4.398261e+07,2026-06-03T23:04:01
95,rl,sdft,2,2,seen,pricing_regret,2.185761e+01,2026-06-03T23:04:01
96,rl,naive,1,1,seen,profit_index,1.000000e+00,2026-06-03T23:04:02
97,rl,naive,1,2,future,profit_index,1.008508e+00,2026-06-03T23:04:02
98,rl,naive,2,1,seen,profit_index,1.014650e+00,2026-06-03T23:04:02
99,rl,naive,2,2,seen,profit_index,1.000000e+00,2026-06-03T23:04:02
100,rl,ewc,1,1,seen,profit_index,1.000000e+00,2026-06-03T23:04:02
101,rl,ewc,1,2,future,profit_index,1.008508e+00,2026-06-03T23:04:02


Logged metrics:
model_type   cl_method  metric_name       
forecasting  ewc        mase                  4
                        rmse                  4
                        smape                 4
             naive      mase                  4
                        rmse                  4
                        smape                 4
             replay     mase                  4
                        rmse                  4
                        smape                 4
             sdft       mase                  4
                        rmse                  4
                        smape                 4
rl           ewc        avg_episode_reward    4
                        cumulative_profit     4
                        pricing_regret        4
                        profit_index          4
             naive      avg_episode_reward    4
                        cumulative_profit     4
                        pricing_regret        4
                        profi

## Optional Summary Tables

In [9]:
cl_summary = build_cl_summary()
tables = print_and_save_comparison_tables(cl_summary)
cl_summary

CL Summary (BWT / FWT):

model_type cl_method primary_metric  avg_final_perf  avg_online_perf     bwt     fwt
forecasting     naive           mase          7.9739           8.6291  1.3105  0.0000
forecasting       ewc           mase          3.4819           3.4836  0.0035 19.2816
forecasting    replay           mase          5.7753           5.7564 -0.0378 18.2214
forecasting      sdft           mase          2.0617           2.0273 -0.0687 22.6879
         rl     naive   profit_index          1.0073           1.0000  0.0146  0.0000
         rl       ewc   profit_index          1.0233           1.0131  0.0205  0.0000
         rl    recall   profit_index          0.9996           0.9802  0.0388  0.0000
         rl      sdft   profit_index          0.9874           0.9818  0.0112  0.0000


  FORECASTING - MASE (lower is better)
           Task 1   Task 2
cl_method                 
ewc        0.8776   6.0896
naive      3.0904  14.1679
replay     1.3883  10.1246
sdft       1.3099   2.7448

  FORECASTING - sMAPE
             Task 1    Task 2
cl_method                    
ewc         74.8005   69.9237
naive      107.6951  109.5287
replay      67.7563   96.9590
sdft        85.4020   52.9304

  RL - PROFIT INDEX (vs naive fresh per-task, higher is better)
           Task 1  Task 2
cl_method                
ewc           1.0  1.0261
naive         1.0  1.0000
recall        1.0  0.9603
sdft          1.0  0.9636

  RL - CUMULATIVE PROFIT (raw MYR, reference)
                 Task 1        Task 2
cl_method                            
ewc        4.204903e+07  4.683546e+07
naive      4.204903e+07  4.564388e+07
recall     4.204903e+07  4.383261e+07
sdft       4.204903e+07  4.398261e+07

  RL - PRICING REGRET (lower is better)
            Task 1   Task 2
cl_method                  
ew

,model_type,cl_method,primary_metric,avg_final_perf,avg_online_perf,bwt,fwt
0,forecasting,naive,mase,7.9739,8.6291,1.3105,0.0000
1,forecasting,ewc,mase,3.4819,3.4836,0.0035,19.2816
2,forecasting,replay,mase,5.7753,5.7564,-0.0378,18.2214
3,forecasting,sdft,mase,2.0617,2.0273,-0.0687,22.6879
4,rl,naive,profit_index,1.0073,1.0000,0.0146,0.0000
5,rl,ewc,profit_index,1.0233,1.0131,0.0205,0.0000
6,rl,recall,profit_index,0.9996,0.9802,0.0388,0.0000
7,rl,sdft,profit_index,0.9874,0.9818,0.0112,0.0000
